In [1068]:
import cirq
import numpy as np


# Circuit Builder

In [1069]:
def build_logic_circuit(register, logical_state="0L", flag_qubit=None, flag_key="flag"):
    """Build the 7-qubit 0L/1L-style circuit used in this notebook."""
    if len(register) < 7:
        raise ValueError("This circuit structure needs at least 7 qubits.")

    circuit = cirq.Circuit()

    init_ops = [cirq.H(register[0]), cirq.H(register[1])]
    if logical_state == "1L" or logical_state == "minus":
        init_ops.append(cirq.X(register[2]))
    init_ops.append(cirq.H(register[3]))
    circuit.append(cirq.Moment(init_ops))

    circuit.append(cirq.Moment([cirq.CNOT(register[2], register[4])]))
    circuit.append(cirq.Moment([cirq.CNOT(register[0], register[6])]))
    circuit.append(cirq.Moment([cirq.CNOT(register[3], register[5])]))
    circuit.append(cirq.Moment([cirq.CNOT(register[2], register[5])]))
    circuit.append(cirq.Moment([cirq.CNOT(register[0], register[4])]))
    circuit.append(cirq.Moment([cirq.CNOT(register[1], register[6])]))
    circuit.append(cirq.Moment([cirq.CNOT(register[0], register[2])]))
    circuit.append(cirq.Moment([cirq.CNOT(register[1], register[5])]))
    circuit.append(cirq.Moment([cirq.CNOT(register[3], register[4])]))
    circuit.append(cirq.Moment([cirq.CNOT(register[1], register[2])]))
    circuit.append(cirq.Moment([cirq.CNOT(register[3], register[6])]))

    if flag_qubit is not None:
        circuit.append(cirq.Moment([cirq.CNOT(register[2], flag_qubit)]))
        circuit.append(cirq.Moment([cirq.CNOT(register[4], flag_qubit)]))
        circuit.append(cirq.Moment([cirq.CNOT(register[5], flag_qubit)]))
        circuit.append(cirq.Moment([cirq.measure(flag_qubit, key=flag_key)]))

    if logical_state == "plus" or logical_state == "minus":
        circuit.append(cirq.Moment([cirq.H(register[0]), cirq.H(register[1]), cirq.H(register[2]), cirq.H(register[3]), cirq.H(register[4]), cirq.H(register[5]), cirq.H(register[6])]))

    return circuit



# Hello Qubit

In [1070]:
# Pick a qubit.
qubit = cirq.GridQubit(0, 0)

# Create a circuit that applies a square root of NOT gate, then measures the qubit.
circuit = cirq.Circuit(cirq.X(qubit) ** 0.5, cirq.measure(qubit, key='m'))
print("Circuit:")
print(circuit)

# Simulate the circuit several times.
simulator = cirq.Simulator()
result = simulator.run(circuit, repetitions=20)
print("Results:")
print(result)

Circuit:
(0, 0): ───X^0.5───M('m')───
Results:
m=10000001110001111001


# 3b)

In [1071]:
register_7 = [cirq.LineQubit(i) for i in range(7)]
circuit_0L = build_logic_circuit(register_7, "0L")
print("Circuit for logical state 0L:")
print(circuit_0L)

Circuit for logical state 0L:
0: ───H───────@───────────@───────@───────────────────
              │           │       │
1: ───H───────┼───────────┼───@───┼───@───────@───────
              │           │   │   │   │       │
2: ───────@───┼───────@───┼───┼───X───┼───────X───────
          │   │       │   │   │       │
3: ───H───┼───┼───@───┼───┼───┼───────┼───@───────@───
          │   │   │   │   │   │       │   │       │
4: ───────X───┼───┼───┼───X───┼───────┼───X───────┼───
              │   │   │       │       │           │
5: ───────────┼───X───X───────┼───────X───────────┼───
              │               │                   │
6: ───────────X───────────────X───────────────────X───


In [1072]:
circuit_1L = build_logic_circuit(register_7, "1L")
print("Circuit for logical state 1L:")
print(circuit_1L)

Circuit for logical state 1L:
0: ───H───────@───────────@───────@───────────────────
              │           │       │
1: ───H───────┼───────────┼───@───┼───@───────@───────
              │           │   │   │   │       │
2: ───X───@───┼───────@───┼───┼───X───┼───────X───────
          │   │       │   │   │       │
3: ───H───┼───┼───@───┼───┼───┼───────┼───@───────@───
          │   │   │   │   │   │       │   │       │
4: ───────X───┼───┼───┼───X───┼───────┼───X───────┼───
              │   │   │       │       │           │
5: ───────────┼───X───X───────┼───────X───────────┼───
              │               │                   │
6: ───────────X───────────────X───────────────────X───


In [1073]:
simulator = cirq.Simulator()
result_0L = simulator.simulate(circuit_0L, qubit_order=register_7)
state_0L = result_0L.final_state_vector

print("Final state vector 0L:")
print(cirq.dirac_notation(state_0L, qid_shape=[2] * len(register_7), decimals=3))

result_1L = simulator.simulate(circuit_1L, qubit_order=register_7)
state_1L = result_1L.final_state_vector

print("Final state vector 1L:")
print(cirq.dirac_notation(state_1L, qid_shape=[2] * len(register_7), decimals=3))


Final state vector 0L:
0.354|0000000⟩ + 0.354|0001111⟩ + 0.354|0110011⟩ + 0.354|0111100⟩ + 0.354|1010101⟩ + 0.354|1011010⟩ + 0.354|1100110⟩ + 0.354|1101001⟩
Final state vector 1L:
0.354|0010110⟩ + 0.354|0011001⟩ + 0.354|0100101⟩ + 0.354|0101010⟩ + 0.354|1000011⟩ + 0.354|1001100⟩ + 0.354|1110000⟩ + 0.354|1111111⟩


# 3c)

In [1074]:
circuit_0L = build_logic_circuit(register_7, "0L", flag_qubit=cirq.LineQubit(7))
circuit_1L = build_logic_circuit(register_7, "1L", flag_qubit=cirq.LineQubit(7))
print("Circuits with measurement:")
print(circuit_0L)
print(circuit_1L)

Circuits with measurement:
0: ───H───────@───────────@───────@───────────────────────────────────────────
              │           │       │
1: ───H───────┼───────────┼───@───┼───@───────@───────────────────────────────
              │           │   │   │   │       │
2: ───────@───┼───────@───┼───┼───X───┼───────X───────@───────────────────────
          │   │       │   │   │       │               │
3: ───H───┼───┼───@───┼───┼───┼───────┼───@───────@───┼───────────────────────
          │   │   │   │   │   │       │   │       │   │
4: ───────X───┼───┼───┼───X───┼───────┼───X───────┼───┼───@───────────────────
              │   │   │       │       │           │   │   │
5: ───────────┼───X───X───────┼───────X───────────┼───┼───┼───@───────────────
              │               │                   │   │   │   │
6: ───────────X───────────────X───────────────────X───┼───┼───┼───────────────
                                                      │   │   │
7: ─────────────────────────────────

In [1075]:
simulator = cirq.Simulator()
result = simulator.run(circuit_0L, repetitions=20)

# Summary counts
print(result.histogram(key='flag'))

simulator = cirq.Simulator()
result = simulator.run(circuit_1L, repetitions=20)

# Summary counts
print(result.histogram(key='flag'))

Counter({0: 20})
Counter({1: 20})


In [1076]:
result = simulator.run(circuit_0L, repetitions=1)
bit = bool(result.measurements['flag'][0][0])
if bit == 0:
    print("0L is the following state:")
    print(cirq.dirac_notation(state_0L, qid_shape=[2] * len(register_7), decimals=3))

result = simulator.run(circuit_1L, repetitions=1)
bit = bool(result.measurements['flag'][0][0])
if bit == 1:
    print("1L is the following state:")
    print(cirq.dirac_notation(state_1L, qid_shape=[2] * len(register_7), decimals=3))

0L is the following state:
0.354|0000000⟩ + 0.354|0001111⟩ + 0.354|0110011⟩ + 0.354|0111100⟩ + 0.354|1010101⟩ + 0.354|1011010⟩ + 0.354|1100110⟩ + 0.354|1101001⟩
1L is the following state:
0.354|0010110⟩ + 0.354|0011001⟩ + 0.354|0100101⟩ + 0.354|0101010⟩ + 0.354|1000011⟩ + 0.354|1001100⟩ + 0.354|1110000⟩ + 0.354|1111111⟩


In [1077]:
circuit_plus = build_logic_circuit(register_7, "plus", flag_qubit=cirq.LineQubit(7))
circuit_minus = build_logic_circuit(register_7, "minus", flag_qubit=cirq.LineQubit(7))

print(circuit_plus)
print(circuit_minus)

0: ───H───────@───────────@───────@───────────────────────────────────────────H───
              │           │       │
1: ───H───────┼───────────┼───@───┼───@───────@───────────────────────────────H───
              │           │   │   │   │       │
2: ───────@───┼───────@───┼───┼───X───┼───────X───────@───────────────────────H───
          │   │       │   │   │       │               │
3: ───H───┼───┼───@───┼───┼───┼───────┼───@───────@───┼───────────────────────H───
          │   │   │   │   │   │       │   │       │   │
4: ───────X───┼───┼───┼───X───┼───────┼───X───────┼───┼───@───────────────────H───
              │   │   │       │       │           │   │   │
5: ───────────┼───X───X───────┼───────X───────────┼───┼───┼───@───────────────H───
              │               │                   │   │   │   │
6: ───────────X───────────────X───────────────────X───┼───┼───┼───────────────H───
                                                      │   │   │
7: ────────────────────────────────

In [1078]:
all_qubits = list(register_7) + [cirq.LineQubit(7)]

result_plus = simulator.simulate(circuit_plus, qubit_order=all_qubits)
flag_bits_plus = np.asarray(result_plus.measurements['flag']).reshape(-1)
bit = int(flag_bits_plus[0])

if bit == 0:
    state_plus = result_plus.final_state_vector
    print("The following state 0 after transversal H is:")
    print(cirq.dirac_notation(state_plus, qid_shape=[2] * len(all_qubits), decimals=3))

result_minus = simulator.simulate(circuit_minus, qubit_order=all_qubits)
flag_bits_minus = np.asarray(result_minus.measurements['flag']).reshape(-1)
bit = int(flag_bits_minus[0])

if bit == 1:
    state_minus = result_minus.final_state_vector
    print("The following state 1 after transversal H is:")
    print(cirq.dirac_notation(state_minus, qid_shape=[2] * len(all_qubits), decimals=3))


The following state 0 after transversal H is:
0.25|00000000⟩ + 0.25|00011110⟩ + 0.25|00101100⟩ + 0.25|00110010⟩ + 0.25|01001010⟩ + 0.25|01010100⟩ + 0.25|01100110⟩ + 0.25|01111000⟩ + 0.25|10000110⟩ + 0.25|10011000⟩ + 0.25|10101010⟩ + 0.25|10110100⟩ + 0.25|11001100⟩ + 0.25|11010010⟩ + 0.25|11100000⟩ + 0.25|11111110⟩
The following state 1 after transversal H is:
0.25|00000001⟩ + 0.25|00011111⟩ - 0.25|00101101⟩ - 0.25|00110011⟩ - 0.25|01001011⟩ - 0.25|01010101⟩ + 0.25|01100111⟩ + 0.25|01111001⟩ - 0.25|10000111⟩ - 0.25|10011001⟩ + 0.25|10101011⟩ + 0.25|10110101⟩ + 0.25|11001101⟩ + 0.25|11010011⟩ - 0.25|11100001⟩ - 0.25|11111111⟩


In [1079]:
def transversal_CNOT(init_1, init_2, register_0, register_1, flags=None):
    """Build a parallel two-register circuit and then apply transversal CNOTs."""
    flag_1 = flags[0] if flags else None
    flag_2 = flags[1] if flags else None

    circuit_0L = build_logic_circuit(register_0, init_1)
    circuit_1L = build_logic_circuit(register_1, init_2)

    combined = cirq.Circuit()
    for i in range(max(len(circuit_0L), len(circuit_1L))):
        ops = []
        if i < len(circuit_0L):
            ops.extend(circuit_0L[i].operations)
        if i < len(circuit_1L):
            ops.extend(circuit_1L[i].operations)
        if ops:
            combined.append(cirq.Moment(ops))

    for j in range(7):
        combined.append(cirq.Moment([cirq.CNOT(register_0[j], register_1[j])]))

    combined.append(cirq.Moment([cirq.CNOT(register_0[2], flag_1)], [cirq.CNOT(register_1[2], flag_2)]))
    combined.append(cirq.Moment([cirq.CNOT(register_0[4], flag_1)], [cirq.CNOT(register_1[4], flag_2)]))
    combined.append(cirq.Moment([cirq.CNOT(register_0[5], flag_1)], [cirq.CNOT(register_1[5], flag_2)]))
    combined.append(cirq.Moment([cirq.measure(flags[0], key="flag_0")], [cirq.measure(flags[1], key="flag_1")]))

    return combined




In [1080]:
register_0 = cirq.LineQubit.range(7)
register_1 = cirq.LineQubit.range(8, 15)
flags = [cirq.LineQubit(7), cirq.LineQubit(15)]

In [1081]:
circuit = transversal_CNOT("0L", "0L", register_0, register_1, flags=flags)
print(circuit)

0: ────H───────@───────────@───────@───────────────────@─────────────────────────────────────────────────────
               │           │       │                   │
1: ────H───────┼───────────┼───@───┼───@───────@───────┼───@─────────────────────────────────────────────────
               │           │   │   │   │       │       │   │
2: ────────@───┼───────@───┼───┼───X───┼───────X───────┼───┼───@───────────────────@─────────────────────────
           │   │       │   │   │       │               │   │   │                   │
3: ────H───┼───┼───@───┼───┼───┼───────┼───@───────@───┼───┼───┼───@───────────────┼─────────────────────────
           │   │   │   │   │   │       │   │       │   │   │   │   │               │
4: ────────X───┼───┼───┼───X───┼───────┼───X───────┼───┼───┼───┼───┼───@───────────┼───@─────────────────────
               │   │   │       │       │           │   │   │   │   │   │           │   │
5: ────────────┼───X───X───────┼───────X───────────┼───┼───┼───┼───┼───┼─

In [1082]:
def check_transversal_flags(control_state, target_state, register_0, register_1, flags=flags):
    circuit = transversal_CNOT(control_state, target_state, register_0, register_1, flags=flags)
    result = simulator.run(circuit, repetitions=1)
    flag_0 = int(result.measurements['flag_0'][0][0])
    flag_1 = int(result.measurements['flag_1'][0][0])
    print(f"{control_state}-{target_state}: {flag_0}L, {flag_1}L")
    return flag_0, flag_1


basis_pairs = [("0L", "0L"), ("0L", "1L"), ("1L", "0L"), ("1L", "1L")]
for control_state, target_state in basis_pairs:
    check_transversal_flags(control_state, target_state, register_0, register_1, flags=flags)


0L-0L: 0L, 0L
0L-1L: 0L, 1L
1L-0L: 1L, 1L
1L-1L: 1L, 0L


In [1083]:
circuit = transversal_CNOT("0L", "0L", register_1, register_0, flags=flags)
print(circuit)

                                                                                   ┌──┐   ┌──┐   ┌──┐
0: ────H───────@───────────@───────@───────────────────X──────────────────────────────────────────────────────────────
               │           │       │                   │
1: ────H───────┼───────────┼───@───┼───@───────@───────┼───X──────────────────────────────────────────────────────────
               │           │   │   │   │       │       │   │
2: ────────@───┼───────@───┼───┼───X───┼───────X───────┼───┼───X─────────────────────@────────────────────────────────
           │   │       │   │   │       │               │   │   │                     │
3: ────H───┼───┼───@───┼───┼───┼───────┼───@───────@───┼───┼───┼───X─────────────────┼────────────────────────────────
           │   │   │   │   │   │       │   │       │   │   │   │   │                 │
4: ────────X───┼───┼───┼───X───┼───────┼───X───────┼───┼───┼───┼───┼───X─────────────┼──────@─────────────────────────
           

In [1084]:
for control_state, target_state in basis_pairs:
    check_transversal_flags(control_state, target_state, register_1, register_0, flags=flags)

0L-0L: 0L, 0L
0L-1L: 0L, 1L
1L-0L: 1L, 1L
1L-1L: 1L, 0L


In [1085]:
parity_check_matrix = np.array([[1, 0, 1, 0, 1, 0, 1],
                                [0, 1, 1, 0, 0, 1, 1],
                                [0, 0, 0, 1, 1, 1, 1],])

def syndromes(measurement_results):
    """Extract the syndrome from the measurement results."""
    syndromes = measurement_results @ parity_check_matrix.T % 2
    return syndromes

print("Parity check matrix:")
print(parity_check_matrix)

result = np.array([[0, 1, 1, 0, 1, 1, 0]])

syndrome = syndromes(result)
print("Syndrome for the measurement results:")
print(syndrome)

Parity check matrix:
[[1 0 1 0 1 0 1]
 [0 1 1 0 0 1 1]
 [0 0 0 1 1 1 1]]
Syndrome for the measurement results:
[[0 1 0]]


In [1086]:
look_up_table = {
    (0, 0, 0): -1,
    (1, 0, 0): 0,
    (0, 1, 0): 1,
    (1, 1, 0): 2,
    (0, 0, 1): 3,
    (1, 0, 1): 4,
    (0, 1, 1): 5,
    (1, 1, 1): 6,
}

In [1087]:
def syndrome_extraction(register, ancilla_start, circuit):
    """Append ancilla-based syndrome extraction with X and Z stabilizers."""

    # Z-stabilizer ancillas
    ancilla_sz1 = cirq.LineQubit(ancilla_start)
    ancilla_sz2 = cirq.LineQubit(ancilla_start + 1)
    ancilla_sz3 = cirq.LineQubit(ancilla_start + 2)
    
    # X-stabilizer ancillas
    ancilla_sx1 = cirq.LineQubit(ancilla_start + 3)
    ancilla_sx2 = cirq.LineQubit(ancilla_start + 4)
    ancilla_sx3 = cirq.LineQubit(ancilla_start + 5)

    # Z-Stabilizers (different qubit combinations for Z parity)
    # sz1
    circuit.append(cirq.Moment([cirq.CNOT(register[0], ancilla_sz1)]))
    circuit.append(cirq.Moment([cirq.CNOT(register[2], ancilla_sz1)]))
    circuit.append(cirq.Moment([cirq.CNOT(register[4], ancilla_sz1)]))
    circuit.append(cirq.Moment([cirq.CNOT(register[6], ancilla_sz1)]))
    circuit.append(cirq.Moment([cirq.measure(ancilla_sz1, key='sz1')]))

    # sz2
    circuit.append(cirq.Moment([cirq.CNOT(register[1], ancilla_sz2)]))
    circuit.append(cirq.Moment([cirq.CNOT(register[2], ancilla_sz2)]))
    circuit.append(cirq.Moment([cirq.CNOT(register[5], ancilla_sz2)]))
    circuit.append(cirq.Moment([cirq.CNOT(register[6], ancilla_sz2)]))
    circuit.append(cirq.Moment([cirq.measure(ancilla_sz2, key='sz2')]))

    # sz3
    circuit.append(cirq.Moment([cirq.CNOT(register[3], ancilla_sz3)]))
    circuit.append(cirq.Moment([cirq.CNOT(register[4], ancilla_sz3)]))
    circuit.append(cirq.Moment([cirq.CNOT(register[5], ancilla_sz3)]))
    circuit.append(cirq.Moment([cirq.CNOT(register[6], ancilla_sz3)]))
    circuit.append(cirq.Moment([cirq.measure(ancilla_sz3, key='sz3')]))

    # X-Stabilizers
    # sx1
    circuit.append(cirq.Moment([cirq.H(ancilla_sx1)]))
    circuit.append(cirq.Moment([cirq.CNOT(ancilla_sx1, register[0])]))
    circuit.append(cirq.Moment([cirq.CNOT(ancilla_sx1, register[2])]))
    circuit.append(cirq.Moment([cirq.CNOT(ancilla_sx1, register[4])]))
    circuit.append(cirq.Moment([cirq.CNOT(ancilla_sx1, register[6])]))
    circuit.append(cirq.Moment([cirq.H(ancilla_sx1)]))
    circuit.append(cirq.Moment([cirq.measure(ancilla_sx1, key='sx1')]))

    # sx2
    circuit.append(cirq.Moment([cirq.H(ancilla_sx2)]))
    circuit.append(cirq.Moment([cirq.CNOT(ancilla_sx2, register[1])]))
    circuit.append(cirq.Moment([cirq.CNOT(ancilla_sx2, register[2])]))
    circuit.append(cirq.Moment([cirq.CNOT(ancilla_sx2, register[5])]))
    circuit.append(cirq.Moment([cirq.CNOT(ancilla_sx2, register[6])]))
    circuit.append(cirq.Moment([cirq.H(ancilla_sx2)]))
    circuit.append(cirq.Moment([cirq.measure(ancilla_sx2, key='sx2')]))

    # sx3
    circuit.append(cirq.Moment([cirq.H(ancilla_sx3)]))
    circuit.append(cirq.Moment([cirq.CNOT(ancilla_sx3, register[3])]))
    circuit.append(cirq.Moment([cirq.CNOT(ancilla_sx3, register[4])]))
    circuit.append(cirq.Moment([cirq.CNOT(ancilla_sx3, register[5])]))
    circuit.append(cirq.Moment([cirq.CNOT(ancilla_sx3, register[6])]))
    circuit.append(cirq.Moment([cirq.H(ancilla_sx3)]))
    circuit.append(cirq.Moment([cirq.measure(ancilla_sx3, key='sx3')]))

    return circuit


In [1088]:
register = cirq.LineQubit.range(7)

circuit_0L = build_logic_circuit(register, "0L")
# Add X error on the second qubit to test syndrome extraction
circuit_0L.append(cirq.Moment([cirq.Y(register[4])]))
circuit_with_syndrome = syndrome_extraction(register, 100, circuit_0L)
print(circuit_with_syndrome)


0: ─────H───────@───────────@───────@───────────────────────@────────────────────────────────────────────────────────────────────────────────────X────────────────────────────────────────────────────────────────────────────────────────────────────
                │           │       │                       │                                                                                    │
1: ─────H───────┼───────────┼───@───┼───@───────@───────────┼──────────────────────────@─────────────────────────────────────────────────────────┼──────────────────────────────────X─────────────────────────────────────────────────────────────────
                │           │   │   │   │       │           │                          │                                                         │                                  │
2: ─────────@───┼───────@───┼───┼───X───┼───────X───────────┼───@──────────────────────┼───@─────────────────────────────────────────────────────┼───X───────────────────────────

In [1089]:
result = simulator.run(circuit_with_syndrome, repetitions=1)
print(circuit_with_syndrome)

# Extract syndrome measurements as 3-vectors
sx1 = int(result.measurements['sx1'][0][0])
sx2 = int(result.measurements['sx2'][0][0])
sx3 = int(result.measurements['sx3'][0][0])
sz1 = int(result.measurements['sz1'][0][0])
sz2 = int(result.measurements['sz2'][0][0])
sz3 = int(result.measurements['sz3'][0][0])

sx_vector = np.array([sx1, sx2, sx3])
sz_vector = np.array([sz1, sz2, sz3])
print("Z-stabilizer syndrome:", sz_vector)
print("X-stabilizer syndrome:", sx_vector)


0: ─────H───────@───────────@───────@───────────────────────@────────────────────────────────────────────────────────────────────────────────────X────────────────────────────────────────────────────────────────────────────────────────────────────
                │           │       │                       │                                                                                    │
1: ─────H───────┼───────────┼───@───┼───@───────@───────────┼──────────────────────────@─────────────────────────────────────────────────────────┼──────────────────────────────────X─────────────────────────────────────────────────────────────────
                │           │   │   │   │       │           │                          │                                                         │                                  │
2: ─────────@───┼───────@───┼───┼───X───┼───────X───────────┼───@──────────────────────┼───@─────────────────────────────────────────────────────┼───X───────────────────────────

In [1090]:
def decoder(sx_vector, sz_vector):
    """Decode the syndrome vectors to identify the error location."""
    # Convert to tuples for lookup
    sx_tuple = tuple(sx_vector)
    sz_tuple = tuple(sz_vector)

    # Look up the error locations using the predefined look-up table
    x_error_location = look_up_table.get(sx_tuple, -1)  # -1 indicates no error
    z_error_location = look_up_table.get(sz_tuple, -1)  # -1 indicates no error

    return x_error_location, z_error_location

print("Decoding the syndrome vectors...")
x_error_location, z_error_location = decoder(sx_vector, sz_vector)
print(f"X error location: {x_error_location}")
print(f"Z error location: {z_error_location}")

Decoding the syndrome vectors...
X error location: 4
Z error location: 4
